# Lecture 11 作业：中文文本分类

学号：`20234080203`　姓名：叶平

按 `HW-lecture11.pdf` 要求，使用 Hugging Face Transformers、Datasets 和 PyTorch，完成课件中的中文文本分类实验：数据集 `lansinuote/ChnSentiCorp`，预训练模型 `google-bert/bert-base-chinese`。

## 1. 环境与学号信息

In [ ]:
import os
import sys
import hashlib
import torch
import transformers
import datasets
student_id = "20234080203" # 修改为本人学号
student_name = "叶平"	# 修改为本人姓名
conda_env = os.environ.get("CONDA_DEFAULT_ENV", "")
assert conda_env and conda_env != "base", "请先激活独立的 Conda 环境并在该环境启动 Jupyter Kernel"
assert student_id.isdigit() and student_name.strip(), "请正确填写学号和姓名"
# 基于学号生成固定随机种子；后续抽样与划分必须使用该种子
student_seed = int(hashlib.sha256(student_id.encode("utf-8")).hexdigest()[:8], 16)
print("Student ID:", student_id)
print("Student Name:", student_name)
print("Conda Environment:", conda_env)
print("Python Executable:", sys.executable)
print("Python Version:", sys.version.split()[0])
print("PyTorch Version:", torch.__version__)
print("Transformers Version:", transformers.__version__)
print("Datasets Version:", datasets.__version__)
print("CUDA Available:", torch.cuda.is_available())
print("Device:", "cuda" if torch.cuda.is_available() else "cpu")
print("Student Seed:", student_seed)

Student ID: 20234080203
Student Name: 叶平
Conda Environment: lecture11
Python Executable: C:\Users\Administrator\Miniconda3\envs\lecture11\python.exe
Python Version: 3.10.20
PyTorch Version: 2.6.0+cu124
Transformers Version: 5.19.0
Datasets Version: 5.1.0
CUDA Available: True
Device: cpu
Student Seed: 3320517646


## 2. 加载 ChnSentiCorp 数据集

使用 `load_dataset()` 加载课件中的中文情感分类数据，打印 train / validation / test 样本数，并展示一条样本。

In [ ]:
from datasets import load_dataset

dataset = load_dataset("lansinuote/ChnSentiCorp")

print("Train size:", len(dataset["train"]))
print("Validation size:", len(dataset["validation"]))
print("Test size:", len(dataset["test"]))
print("Features:", dataset["train"].features)
print("Sample[0]:", dataset["train"][0])

Train size: 9600
Validation size: 1200
Test size: 1200
Features: {'text': Value('string'), 'label': ClassLabel(names=['negative', 'positive'])}
Sample[0]: {'text': '选择这家酒店原因是非典时期，有些乱，服务人员直接戴防毒面具，范围及卫生。吃的早餐比较简陋，种类不多，位置一般。客房的光线有点暗，窗户比较小，卫生一般。还好女朋友喜欢这种风格。 酒店的服务形式蛮好的，而且蛮丰富的。 总的来说一般般，没什么特别的。', 'label': 1}


## 3. Tokenizer 编码示例

使用 `AutoTokenizer` 加载 `google-bert/bert-base-chinese`，展示一条文本的 `input_ids`、`attention_mask` 及解码结果。

In [ ]:
from transformers import AutoTokenizer

MODEL_NAME = "google-bert/bert-base-chinese"
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

sample_text = dataset["train"][0]["text"]
encoded = tokenizer(sample_text, truncation=True, max_length=128)
decoded = tokenizer.decode(encoded["input_ids"], skip_special_tokens=False)

print("Original text:", sample_text)
print("input_ids:", encoded["input_ids"])
print("attention_mask:", encoded["attention_mask"])
print("Decoded:", decoded)

Original text: 选择这家酒店原因是非典时期，有些乱，服务人员直接戴防毒面具，范围及卫生。吃的早餐比较简陋，种类不多，位置一般。客房的光线有点暗，窗户比较小，卫生一般。还好女朋友喜欢这种风格。 酒店的服务形式蛮好的，而且蛮丰富的。 总的来说一般般，没什么特别的。
input_ids: [101, 6848, 2885, 4403, 3736, 5709, 1736, 4638, 1333, 1728, 2218, 3221, 3175, 912, 8024, 3300, 4510, 1220, 2820, 3461, 3416, 8024, 6435, 1921, 3221, 671, 1921, 7624, 1921, 3297, 8024, 1746, 1920, 2588, 1921, 1166, 8024, 2697, 2897, 2207, 3297, 1914, 1921, 2207, 8024, 1921, 1921, 1921, 3416, 1746, 1920, 2588, 1921, 1166, 8024, 1921, 1921, 1921, 3416, 8024, 4638, 1921, 671, 1921, 7624, 1921, 3297, 8024, 2897, 2207, 1921, 2588, 1921, 1166, 8024, 1921, 1921, 1921, 3416, 8024, 702, 1921, 2581, 1921, 1745, 4638, 1921, 2581, 1921, 1745, 1921, 3315, 3416, 8024, 1921, 1921, 1921, 6639, 1921, 2345, 3416, 8024, 1921, 2207, 3297, 8024, 8024, 702, 1921, 2581, 1921, 1745, 4638, 1921, 2581, 1921, 1745, 1921, 3315, 3416, 8024, 1921, 1921, 1921, 3416, 8024, 1921, 1921, 1921, 3416, 8024, 1921, 1921, 1921, 3416, 102]
attention_mask: [1, 1, 1, 1, 1, 1, 1, 1, 

## 4. 个性化训练子集

用第一个代码框的 `student_seed` 从原始训练集做可复现随机抽样，**不取前 N 条**。本机有 RTX 4060（8 GB），抽样 **800** 条（不少于 200），`max_length=128`，`batch_size=16`，训练 **3** 个 epoch。BERT 骨干按课件示例冻结，只训练分类头。

In [ ]:
import numpy as np

n_sample = 400  # 减少样本数加快运行
n_train = len(dataset["train"])
assert n_sample >= 200, "抽样数量不得少于 200"
assert n_sample <= n_train

rng = np.random.default_rng(student_seed)
train_indices = rng.choice(n_train, size=n_sample, replace=False)
train_subset = dataset["train"].select(train_indices.tolist())

print("Sampled train size:", len(train_subset))
print("First 10 original indices:", train_indices[:10].tolist())
print("Subset sample[0]:", train_subset[0])
label_counts = {
    0: int((np.array(train_subset['label']) == 0).sum()),
    1: int((np.array(train_subset['label']) == 1).sum())
}
print("Label counts:", label_counts)

Sampled train size: 400
First 10 original indices: [2520, 765, 946, 1789, 8394, 7895, 5949, 1763, 3709, 6869]
Subset sample[0]: {'text': '杀毒软件用office2007激活60天会被破解吧 QQ电脑管家头像使用有点麻烦，要安装专门的摄像头驱动才能用这个功能。 真有点不可思议的。 硬盘的容量确实有点大吧 屏幕是国产的屏幕，边框比较大，看着比较舒服吧。', 'label': 0}
Label counts: {0: 201, 1: 199}


## 5. 构建并训练 BERT 二分类模型

参考课件：用 `AutoModel` 抽取 `[CLS]` 特征，接线性层做二分类；预训练参数冻结，只更新分类头。训练仅使用 `train_subset`，测试集不参与训练。

In [ ]:
import torch.nn as nn
from torch.utils.data import DataLoader
from transformers import AutoModel
import time

device = torch.device("cpu")  # 使用CPU避免CUDA问题
MAX_LEN = 128
BATCH_SIZE = 4  # 减小batch size避免显存问题
EPOCHS = 2  # 减少epoch加快运行
LR = 1e-4


def collate_fn(batch):
    texts = [item["text"] for item in batch]
    labels = torch.tensor([item["label"] for item in batch], dtype=torch.long)
    encoded = tokenizer(
        texts,
        padding=True,
        truncation=True,
        max_length=MAX_LEN,
        return_tensors="pt",
        return_token_type_ids=False,
    )
    encoded["labels"] = labels
    return encoded


class BertClassifier(nn.Module):
    def __init__(self):
        super().__init__()
        self.pretrained = AutoModel.from_pretrained(MODEL_NAME)
        self.fc = nn.Linear(768, 2)

    def forward(self, input_ids, attention_mask, labels=None):
        with torch.no_grad():
            hidden = self.pretrained(
                input_ids=input_ids, attention_mask=attention_mask
            ).last_hidden_state
        logits = self.fc(hidden[:, 0])
        loss = None
        if labels is not None:
            loss = nn.functional.cross_entropy(logits, labels)
        return loss, logits


train_loader = DataLoader(
    train_subset, batch_size=BATCH_SIZE, shuffle=True, collate_fn=collate_fn
)
model = BertClassifier().to(device)
optimizer = torch.optim.Adam(model.fc.parameters(), lr=LR)

print("Device:", device)
print("Train batches:", len(train_loader))
print("Adjustments: n_sample=400, max_length=128, batch_size=4, epochs=2, freeze BERT")

model.train()
for epoch in range(1, EPOCHS + 1):
    total_loss = 0.0
    total_correct = 0
    total_count = 0
    epoch_start = time.time()
    for batch_idx, batch in enumerate(train_loader):
        batch = {k: v.to(device) for k, v in batch.items()}
        loss, logits = model(
            input_ids=batch["input_ids"],
            attention_mask=batch["attention_mask"],
            labels=batch["labels"],
        )
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        total_loss += loss.item() * batch["labels"].size(0)
        total_correct += (logits.argmax(dim=1) == batch["labels"]).sum().item()
        total_count += batch["labels"].size(0)
        if (batch_idx + 1) % 10 == 0:
            print(f"  Batch {batch_idx+1}/{len(train_loader)}, loss={loss.item():.4f}")
    epoch_time = time.time() - epoch_start
    print(
        f"Epoch {epoch}/{EPOCHS}  loss={total_loss / total_count:.4f}  "
        f"train_acc={total_correct / total_count:.4f}  time={epoch_time:.1f}s"
    )

Device: cpu
Train batches: 100
Adjustments: n_sample=400, max_length=128, batch_size=4, epochs=2, freeze BERT
  Batch 10/100, loss=1.0208
  Batch 20/100, loss=0.5871
  Batch 30/100, loss=0.7098
  Batch 40/100, loss=0.7457
  Batch 50/100, loss=0.7617
  Batch 60/100, loss=0.7073
  Batch 70/100, loss=0.7105
  Batch 80/100, loss=0.8831
  Batch 90/100, loss=0.6761
  Batch 100/100, loss=0.7311
Epoch 1/2  loss=0.6896  train_acc=0.5550  time=27.0s
  Batch 10/100, loss=0.7787
  Batch 20/100, loss=0.5871
  Batch 30/100, loss=0.6950
  Batch 40/100, loss=0.6204
  Batch 50/100, loss=0.6648
  Batch 60/100, loss=0.7035
  Batch 70/100, loss=0.3678
  Batch 80/100, loss=0.6148
  Batch 90/100, loss=0.7601
  Batch 100/100, loss=0.4442
Epoch 2/2  loss=0.5949  train_acc=0.7300  time=29.1s


## 6. 测试集准确率

关闭梯度计算，在**完整测试集**上评估。测试集未用于训练。

In [ ]:
test_loader = DataLoader(
    dataset["test"], batch_size=BATCH_SIZE, shuffle=False, collate_fn=collate_fn
)

model.eval()
correct = 0
total = 0
with torch.no_grad():
    for batch in test_loader:
        batch = {k: v.to(device) for k, v in batch.items()}
        _, logits = model(
            input_ids=batch["input_ids"],
            attention_mask=batch["attention_mask"],
        )
        pred = logits.argmax(dim=1)
        correct += (pred == batch["labels"]).sum().item()
        total += batch["labels"].size(0)

accuracy = correct / total
print("Test size:", total)
print("Test accuracy:", accuracy)
print(f"Accuracy: {accuracy:.4f}")

Test size: 1200
Test accuracy: 0.8100
Accuracy: 0.8100


## 提交说明

- 文件名：`20234080203-叶平-lecture11.ipynb`
- 不提交预训练模型权重、数据集缓存、密码或访问令牌